# Stim to QIR Compilation & Simulation

This notebook demonstrates compiling a [Stim](https://github.com/quantumlib/Stim) circuit to QIR and running it on the simulator using the `qdk.stim` module.

The two entry points are:

- `qdk.stim.compile(src, noise)` → returns a `(qir, noise)` tuple. The QIR is a string; `noise` is a `NoiseConfig` (auto-built from the circuit's correlated-error chains when you pass `None`).
- `qdk.stim.run(src, shots=..., noise=..., seed=..., type=...)` → compiles and simulates in one step, returning a list of per-shot measurement results.

In [1]:
from qdk import stim

## Basics

Compile a plain Clifford circuit to QIR, then run it on the simulator.

In [2]:
# Start simple: a Clifford circuit with no noise.
# compile() returns a (qir, noise) tuple. Pass noise=None to auto-build a
# NoiseConfig (noiseless here, since there are no error channels).
stim_circuit = """H 0
CZ 0 1
H 0
MR 0 1
"""

qir, noise = stim.compile(stim_circuit, None)
print(qir)

define i64 @ENTRYPOINT__main() #0 {
  call void @__quantum__rt__initialize(ptr null)
  call void @__quantum__qis__h__body(ptr inttoptr (i64 0 to ptr))
  call void @__quantum__qis__cz__body(ptr inttoptr (i64 0 to ptr), ptr inttoptr (i64 1 to ptr))
  call void @__quantum__qis__h__body(ptr inttoptr (i64 0 to ptr))
  call void @__quantum__qis__mresetz__body(ptr inttoptr (i64 0 to ptr), ptr inttoptr (i64 0 to ptr))
  call void @__quantum__qis__mresetz__body(ptr inttoptr (i64 1 to ptr), ptr inttoptr (i64 1 to ptr))
  call void @__quantum__rt__array_record_output(i64 2, ptr null)
  call void @__quantum__rt__result_record_output(ptr inttoptr (i64 0 to ptr), ptr null)
  call void @__quantum__rt__result_record_output(ptr inttoptr (i64 1 to ptr), ptr null)
  ret i64 0
}

declare void @__quantum__qis__mresetz__body(ptr, ptr)
declare void @__quantum__rt__result_record_output(ptr, ptr)
declare void @__quantum__rt__array_record_output(i64, ptr)
declare void @__quantum__qis__cz__body(ptr, ptr)
declare

In [3]:
stim_circuit = """H 0
CX 0 1
MR 0 1
"""

# run() compiles and simulates in one step.
results = stim.run(stim_circuit, shots=100, type="cpu")
print(f"Got {len(results)} shots")
for r in results[:10]:
    print(r)

Got 100 shots
[One, Zero]
[Zero, Zero]
[One, Zero]
[One, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[One, Zero]
[Zero, Zero]


## Preselection

The custom `#!preselect_begin` / `#!preselect_expect` annotations compile to a
retry loop in the QIR: the block is repeated until the measured result matches
the expected value. This is useful for post-selecting on a desired measurement
outcome.

In [4]:
preselect_circuit = """H 0
MR 0
#!preselect_begin
H 0
MR 0
#!preselect_expect 0 1
H 0
CZ 0 1
MR 0 1
"""

qir, noise = stim.compile(preselect_circuit, None)
print(qir)

results = stim.run(preselect_circuit, shots=50, type="cpu")
for r in results[:10]:
    print(r)

define i64 @ENTRYPOINT__main() #0 {
  call void @__quantum__rt__initialize(ptr null)
  call void @__quantum__qis__h__body(ptr inttoptr (i64 0 to ptr))
  call void @__quantum__qis__mresetz__body(ptr inttoptr (i64 0 to ptr), ptr inttoptr (i64 0 to ptr))
  br label %preselect_begin_0
preselect_begin_0:
  call void @__quantum__qis__h__body(ptr inttoptr (i64 0 to ptr))
  call void @__quantum__qis__mresetz__body(ptr inttoptr (i64 0 to ptr), ptr inttoptr (i64 1 to ptr))
  %preselect_r0 = call i1 @__quantum__rt__read_result(ptr inttoptr (i64 0 to ptr))
  br i1 %preselect_r0, label %preselect_continue_0, label %preselect_begin_0
preselect_continue_0:
  call void @__quantum__qis__h__body(ptr inttoptr (i64 0 to ptr))
  call void @__quantum__qis__cz__body(ptr inttoptr (i64 0 to ptr), ptr inttoptr (i64 1 to ptr))
  call void @__quantum__qis__mresetz__body(ptr inttoptr (i64 0 to ptr), ptr inttoptr (i64 2 to ptr))
  call void @__quantum__qis__mresetz__body(ptr inttoptr (i64 1 to ptr), ptr inttoptr (i

In [ ]:
## Noise channels

From here on we exercise the noise channels. Each one compiles to a noise
intrinsic plus a `NoiseTable` that maps Pauli/loss strings to probabilities.
Passing `noise=None` to `stim.compile` auto-builds the `NoiseConfig` from the
circuit and returns it, and `stim.run` wires it into the simulator.

### Correlated error

A `CORRELATED_ERROR` / `ELSE_CORRELATED_ERROR` chain is a set of mutually
exclusive faults applied together as one correlated event.

In [ ]:
# A correlated error on qubits 0 and 1: X on both, together, with probability 0.2.
correlated_circuit = """CORRELATED_ERROR(0.2) X0 X1
ELSE_CORRELATED_ERROR(0.1) X1 X2
ELSE_CORRELATED_ERROR(0.1) X0 X2
MR 0 1
"""

results = stim.run(correlated_circuit, shots=20, type="cpu")
for r in results:
    print(r)

[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[One, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[One, One]
[Zero, Zero]
[One, One]
[One, Zero]
[One, Zero]
[Zero, Zero]
[One, Zero]
[Zero, Zero]
[One, One]
[One, One]
[One, One]
[One, One]


### Single-qubit Pauli errors: `X_ERROR`, `Y_ERROR`, `Z_ERROR`

Unlike `CORRELATED_ERROR`, these apply an **independent** Pauli to each target
qubit with the given probability. Each target becomes its own one-qubit noise
table. An `X_ERROR` before a `Z`-basis measurement flips the result.

In [25]:
# Independent X errors on qubits 0 and 1, each with probability 0.1.
# Expect ~10% of each qubit to flip to 1, independently.
xyz_circuit = """X_ERROR(0.1) 0 1
MR 0 1
"""

results = stim.run(xyz_circuit, shots=20, type="cpu")

for r in results:    print(r)

[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[One, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, One]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[One, Zero]
[Zero, Zero]


### Qubit loss: `LOSS_ERROR`

`LOSS_ERROR` independently loses each target qubit with the given probability.
A lost qubit is reported as `L` in the measurement results (rather than `0`/`1`).

In [32]:
# Independent loss on qubits 0 and 1, each with probability 0.15.
# Lost qubits show up as `L`; otherwise the qubit measures 0.
loss_circuit = """LOSS_ERROR(0.15) 0 1
MR 0 1
"""

results = stim.run(loss_circuit, shots=20, type="cpu")

for r in results:    print(r)

[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]


### Loss inside a correlated error

A `CORRELATED_ERROR` chain can mix Pauli terms (`X`/`Y`/`Z`) and loss terms
(`L`) in the same fault. Here the mutually-exclusive branches lose qubit 0,
lose qubit 1, or lose qubit 0 while applying `Z` to qubit 1.

In [39]:
# Mutually-exclusive correlated branches mixing loss and Pauli terms.
mixed_circuit = """CORRELATED_ERROR(0.1) L0
ELSE_CORRELATED_ERROR(0.1) L1
ELSE_CORRELATED_ERROR(0.1) L0 Z1
MR 0 1
"""

results = stim.run(mixed_circuit, shots=20, type="cpu")

for r in results:    print(r)

[Zero, Zero]
[Zero, Zero]
[Loss, Zero]
[Zero, Zero]
[Zero, Zero]
[Loss, Zero]
[Zero, Loss]
[Zero, Zero]
[Zero, Zero]
[Zero, Loss]
[Zero, Zero]
[Zero, Zero]
[Loss, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]


### Depolarizing noise: `DEPOLARIZE1`, `DEPOLARIZE2`

`DEPOLARIZE1(p)` applies one of the 3 single-qubit Paulis (each with
probability `p/3`) independently to each target. `DEPOLARIZE2(p)` applies one
of the 15 non-identity two-qubit Paulis (each with probability `p/15`) to each
qubit pair.

In [ ]:
# DEPOLARIZE1: single-qubit depolarizing on qubits 0 and 1.
# In the Z basis, X and Y components flip the measurement, so ~2/3 * p flip.
print("DEPOLARIZE1(0.3):")
depolarize1_circuit = """DEPOLARIZE1(0.3) 0 1
MR 0 1
"""
for r in stim.run(depolarize1_circuit, shots=20, type="cpu"):
    print(r)

print("\nDEPOLARIZE2(0.3):")
depolarize2_circuit = """DEPOLARIZE2(0.3) 0 1
MR 0 1
"""

for r in stim.run(depolarize2_circuit, shots=20, type="cpu"):    print(r)

DEPOLARIZE1(0.3):
[Zero, Zero]
[Zero, Zero]
[Zero, One]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[One, Zero]
[Zero, One]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, One]
[Zero, One]
[One, One]
[Zero, Zero]
[Zero, One]
[Zero, Zero]

DEPOLARIZE2(0.3):
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, One]
[Zero, Zero]
[Zero, One]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[Zero, Zero]
[One, One]
[One, One]
[One, Zero]
[One, Zero]
